In [4]:
from pathlib import Path

# 여기를 실제 변환된 데이터셋 경로로 바꾸세요.
dataset_root = Path(r"/home/jack/flower_vla_calvin/custom_ft_dataset_raw_action")

# 'training', 'validation', 'all' 중 하나
split = "training"

# 앞에서 몇 개 액션을 샘플로 출력할지
sample_count = 10

# gripper(last dim) 값을 몇 자리까지 반올림해서 빈도 요약할지
gripper_round = 4

In [5]:
import json
from typing import Iterable, List, Sequence, Tuple

import numpy as np


def resolve_split_dirs(dataset_root: Path, split: str) -> List[Path]:
    if split == "all":
        split_dirs = [dataset_root / "training", dataset_root / "validation"]
    else:
        split_dirs = [dataset_root / split]

    missing = [path for path in split_dirs if not path.is_dir()]
    if missing:
        names = ", ".join(str(path) for path in missing)
        raise FileNotFoundError(f"Split directory not found: {names}")
    return split_dirs


def iter_episode_files(split_dir: Path) -> Iterable[Path]:
    yield from sorted(split_dir.glob("episode_*.npz"))


def load_action(npz_path: Path) -> np.ndarray:
    with np.load(npz_path) as data:
        if "rel_actions" in data:
            action = data["rel_actions"]
        elif "actions" in data:
            action = data["actions"]
        else:
            keys = ", ".join(sorted(data.files))
            raise KeyError(f"{npz_path} does not contain rel_actions/actions. Keys: {keys}")

    return np.asarray(action, dtype=np.float32).reshape(-1)


def format_vector(values: Sequence[float]) -> str:
    return "[" + ", ".join(f"{value: .6f}" for value in values) + "]"


def describe_episode_lengths(split_dir: Path) -> None:
    ep_path = split_dir / "ep_start_end_ids.npy"
    if not ep_path.is_file():
        return

    ep_start_end_ids = np.load(ep_path)
    if ep_start_end_ids.size == 0:
        print("episodes: 0")
        return

    lengths = ep_start_end_ids[:, 1] - ep_start_end_ids[:, 0]
    print(
        "episodes:",
        len(lengths),
        f"(min_len={int(lengths.min())}, mean_len={float(lengths.mean()):.2f}, max_len={int(lengths.max())})",
    )


def describe_conversion_metadata(split_dir: Path) -> None:
    metadata_path = split_dir / "conversion_metadata.json"
    if not metadata_path.is_file():
        return

    with metadata_path.open("r", encoding="utf-8") as handle:
        metadata = json.load(handle)

    robot_obs_dim = metadata.get("robot_obs_dim")
    saved_steps = metadata.get("saved_step_count")
    saved_episodes = metadata.get("saved_episode_count")
    print(
        "metadata:",
        f"saved_episodes={saved_episodes}, saved_steps={saved_steps}, robot_obs_dim={robot_obs_dim}",
    )


def inspect_split(split_dir: Path, sample_count: int = 10, gripper_round: int = 4) -> None:
    files = list(iter_episode_files(split_dir))
    if not files:
        print(f"\n[{split_dir.name}]")
        print("No episode_*.npz files found.")
        return

    count = 0
    action_dim = None
    sum_vec = None
    sumsq_vec = None
    min_vec = None
    max_vec = None
    samples: List[Tuple[Path, np.ndarray]] = []
    gripper_values: List[float] = []

    for npz_path in files:
        action = load_action(npz_path)

        if action_dim is None:
            action_dim = action.shape[0]
            sum_vec = np.zeros(action_dim, dtype=np.float64)
            sumsq_vec = np.zeros(action_dim, dtype=np.float64)
            min_vec = np.full(action_dim, np.inf, dtype=np.float64)
            max_vec = np.full(action_dim, -np.inf, dtype=np.float64)

        if action.shape[0] != action_dim:
            raise ValueError(
                f"Inconsistent action dimension in {npz_path}: expected {action_dim}, got {action.shape[0]}"
            )

        action64 = action.astype(np.float64)
        sum_vec += action64
        sumsq_vec += action64 * action64
        min_vec = np.minimum(min_vec, action64)
        max_vec = np.maximum(max_vec, action64)
        gripper_values.append(float(action64[-1]))

        if len(samples) < sample_count:
            samples.append((npz_path, action.copy()))

        count += 1

    mean_vec = sum_vec / count
    var_vec = np.maximum((sumsq_vec / count) - np.square(mean_vec), 0.0)
    std_vec = np.sqrt(var_vec)

    rounded_gripper = np.round(np.asarray(gripper_values), gripper_round)
    unique_vals, unique_counts = np.unique(rounded_gripper, return_counts=True)
    order = np.argsort(unique_counts)[::-1]

    print(f"\n[{split_dir.name}]")
    print(f"num_steps: {count}")
    print(f"action_dim: {action_dim}")
    describe_episode_lengths(split_dir)
    describe_conversion_metadata(split_dir)
    print(f"min : {format_vector(min_vec)}")
    print(f"max : {format_vector(max_vec)}")
    print(f"mean: {format_vector(mean_vec)}")
    print(f"std : {format_vector(std_vec)}")

    print("gripper(last dim) frequency summary:")
    top_k = min(10, len(unique_vals))
    for rank in range(top_k):
        idx = order[rank]
        value = unique_vals[idx]
        freq = unique_counts[idx]
        ratio = freq / count
        print(f"  value={value:.{gripper_round}f}, count={freq}, ratio={ratio:.4f}")

    print(f"first {len(samples)} action samples:")
    for sample_path, action in samples:
        print(f"  {sample_path.name}: {format_vector(action)}")

In [6]:
for split_dir in resolve_split_dirs(dataset_root, split):
    inspect_split(split_dir, sample_count=sample_count, gripper_round=gripper_round)


[training]
num_steps: 16251
action_dim: 7
episodes: 90 (min_len=122, mean_len=180.57, max_len=292)
metadata: saved_episodes=90, saved_steps=16251, robot_obs_dim=13
min : [-0.235410, -0.441333, -0.357910, -0.624257, -0.443157, -0.183310, -0.996993]
max : [ 0.545830,  0.567421,  0.970176,  0.152162,  0.510250,  0.072995,  0.993306]
mean: [ 0.020648,  0.012358,  0.000212, -0.012652, -0.004924, -0.007538,  0.000005]
std : [ 0.081143,  0.104777,  0.116998,  0.070719,  0.073408,  0.019047,  0.093526]
gripper(last dim) frequency summary:
  value=0.0000, count=16071, ratio=0.9889
  value=0.9582, count=2, ratio=0.0001
  value=0.9686, count=2, ratio=0.0001
  value=-0.8152, count=2, ratio=0.0001
  value=-0.8232, count=1, ratio=0.0001
  value=-0.8248, count=1, ratio=0.0001
  value=-0.8273, count=1, ratio=0.0001
  value=-0.8324, count=1, ratio=0.0001
  value=-0.8375, count=1, ratio=0.0001
  value=-0.8393, count=1, ratio=0.0001
first 10 action samples:
  episode_0000000.npz: [-0.007827,  0.000000, 